# ImparTail — chạy đánh giá trên Kaggle

Đọc thẳng dataset có sẵn trên Kaggle, **không tải video, không tách khung hình**:

| Dataset | Nội dung |
|---|---|
| `phmngcduy/action-genome` | annotation (`object_bbox_and_relationship.pkl`, `person_bbox.pkl`, `frame_list.txt`) |
| `phmngcduy/action-genome-frames-part-1..6` | khung hình đã tách, cấu trúc `<video>.mp4/<frame>.png` |

Tác giả có phát hành **checkpoint đã huấn luyện** nên chỉ cần chạy đánh giá.

## Ba rào cản thật và cách xử lý

1. **`torch==1.10.1+cu111`** (bản 2021) — GPU Kaggle hiện tại dùng CUDA 12, không
   có bản dựng cu111. Notebook thử torch có sẵn trước; ô chẩn đoán sẽ nói rõ nếu
   không tương thích thay vì lỗi mơ hồ.
2. **CUDA extension của fasterRCNN** (`nms`, `roi_align`, `roi_crop`) — phải biên
   dịch, và mã sinh từ thời torch 1.x thường không dựng được trên torch 2.x.
3. **`draw_rectangles`** là Cython, phải `build_ext` trước khi import.

`nms` chỉ dùng ở nhánh **sgdet**, nhưng `lib/object_detector.py:28` nạp
`fasterRCNN.resnet` ngay trong `__init__` nên **PREDCLS cũng không tránh được**.

**Đánh giá thành thật:** khả năng chạy trót lọt là thấp. Notebook này dựng để
*kiểm chứng điều đó bằng thực nghiệm* thay vì suy đoán — mỗi ô báo rõ hỏng ở đâu.

In [ ]:
# ── 0. Môi trường thật đang có gì ───────────────────────────────────
import torch, sys, subprocess, os
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("CUDA của torch:", torch.version.cuda)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
print("compute cap :", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "-")
print("arch hỗ trợ :", torch.cuda.get_arch_list() if torch.cuda.is_available() else "-")
r = subprocess.run(["nvcc","--version"], capture_output=True, text=True)
print("nvcc        :", (r.stdout.strip().split("\n")[-1] if r.returncode==0 else "không có nvcc"))

In [ ]:
# ── 0b. Hạ torch cho khớp GPU ───────────────────────────────────────
# Log lần chạy trước: Kaggle cấp Tesla P100 (sm_60) nhưng cài sẵn torch 2.10
# chỉ hỗ trợ từ sm_70. Cùng lỗi đã gặp với APKGC, khắc phục bằng torch 2.4.1.
#
# ImparTail vốn viết cho torch 1.10+cu111 (2021). Không có bản cu111 nào chạy
# được trên CUDA 12, nên 2.4.1+cu121 là điểm gần nhất vừa hợp P100 vừa còn tồn
# tại. CUDA extension của fasterRCNN có dựng nổi trên API torch 2.4 hay không
# thì chưa biết — đó chính là thứ lần chạy này kiểm.
import torch, subprocess, sys
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
need = f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list() if torch.cuda.is_available() else False
print(f"GPU sm_{cap[0]}{cap[1]} | torch {torch.__version__} | cần hạ: {need}")

if need:
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3
    r = subprocess.run([sys.executable, "-c",
        "import torch;a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__);print('arch',torch.cuda.get_arch_list());"
        "print('matmul OK',float((a@a).sum()))"], capture_output=True, text=True)
    print(r.stdout or r.stderr[-500:])
    print("\n>>> torch đã đổi. Các ô sau chạy qua subprocess nên nhận bản mới.")
else:
    print("không cần hạ.")

In [ ]:
# ── 1. Dataset: dò theo tệp mốc, không đoán đường dẫn ───────────────
import os, glob
ROOT = "/kaggle/input"
def find(marker, isdir=False):
    for dp, dn, fn in os.walk(ROOT):
        if (marker in dn) if isdir else (marker in fn):
            return dp
    return None

ANN = find("object_bbox_and_relationship.pkl")
print("annotation :", ANN)
if ANN: print("  ", sorted(os.listdir(ANN)))

# khung hình: thư mục cha chứa nhiều thư mục con dạng <id>.mp4
# Dò ĐỆ QUY. Bản trước chỉ quét 2 mức (ROOT/* và ROOT/*/*) nên trượt: Kaggle
# gắn dataset ở /kaggle/input/datasets/<user>/<slug>/... — sâu 3-4 mức. Cùng
# lần chạy đó annotation tìm thấy (nhờ os.walk) còn khung hình thì không.
FRAME_DIRS = []
for dp, dn, fn in os.walk(ROOT):
    kids = [k for k in dn if k.endswith(".mp4")]
    if len(kids) > 10:
        FRAME_DIRS.append((dp, len(kids)))
        dn[:] = []          # không đi sâu vào 1600 thư mục con
for p, n in FRAME_DIRS:
    print(f"khung hình : {p}  ({n} video)")
print(f"\nTỔNG: {sum(n for _, n in FRAME_DIRS)} thư mục video")
if not (ANN and FRAME_DIRS):
    print("\nCÂY /kaggle/input (3 mức đầu):")
    for dp, dn, fn in os.walk(ROOT):
        depth = dp.replace(ROOT, "").count("/")
        if depth > 3: dn[:] = []; continue
        print("  " * depth, os.path.basename(dp) or ROOT,
              f"[{len(dn)} thư mục, {len(fn)} tệp]")
    raise SystemExit("thiếu dataset")

In [ ]:
# ── 2. Gộp các part khung hình thành một cây thư mục bằng symlink ───
# /kaggle/input chỉ đọc và mỗi part là một mount riêng, nên ImparTail không
# thấy được toàn bộ. Symlink rẻ hơn copy 60GB rất nhiều.
import os
AG = "/kaggle/working/data/ag"
os.makedirs(f"{AG}/frames", exist_ok=True)
os.makedirs(f"{AG}/annotations", exist_ok=True)

n = 0
for src, _ in FRAME_DIRS:
    for v in os.listdir(src):
        if not v.endswith(".mp4"): continue
        dst = f"{AG}/frames/{v}"
        if not os.path.exists(dst):
            os.symlink(f"{src}/{v}", dst); n += 1
print(f"đã liên kết {n} thư mục video -> {AG}/frames")

for f in os.listdir(ANN):
    dst = f"{AG}/annotations/{f}"
    if not os.path.exists(dst): os.symlink(f"{ANN}/{f}", dst)
print("annotations:", sorted(os.listdir(f"{AG}/annotations")))

In [ ]:
# ── 3. Mã nguồn ImparTail ───────────────────────────────────────────
%cd /kaggle/working
!rm -rf ImparTail
!git clone -q https://github.com/rohithpeddi/ImparTail.git
%cd /kaggle/working/ImparTail
import os
print("có:", [f for f in os.listdir(".") if f.endswith((".py",".sh"))][:6])

In [ ]:
# ── 4. Cython: CẢ HAI extension ────────────────────────────────────
# Repo có hai .pyx phải build, không phải một. Bỏ sót bbox thì script chết ở
#   ModuleNotFoundError: No module named 'lib.fpn.box_intersections_cpu.bbox'
# nhưng mãi sau khi đã qua hết phần vá — nên rất tốn một vòng chạy.
# Danh sách này lấy bằng `find . -name "*.pyx"` ở máy, không phải theo lỗi.
import subprocess, sys, os, glob
ROOT = "/kaggle/working/ImparTail"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"],
               capture_output=True, text=True)

# ── Bí danh numpy đã bị gỡ ──────────────────────────────────────────
# np.float / np.bool bị NumPy gỡ hẳn ở 1.24. Quét toàn repo: 13 chỗ / 8 tệp.
# PHẢI vá TRƯỚC khi cythonize — lỗi nằm trong bbox.pyx và chỉ lộ lúc import
# module đã biên dịch, nên vá sau thì vô ích.
#!/usr/bin/env python3
"""Thay các bí danh numpy đã bị gỡ (np.float, np.bool, …).

NumPy gỡ hẳn `np.float`, `np.int`, `np.bool`, `np.object`, `np.str` ở bản 1.24
(deprecated từ 1.20). Mã ImparTail viết trước đó nên còn dùng, và script đánh giá
dừng ở:

    AttributeError: module 'numpy' has no attribute 'float'

Đây chỉ là bí danh của kiểu dựng sẵn Python — thay bằng chính kiểu đó là tương
đương tuyệt đối, không đổi ngữ nghĩa hay độ chính xác.

Quét toàn repo cho ra 13 chỗ trong 8 tệp, gồm cả `.pyx` (phải vá TRƯỚC khi
cythonize, vì lỗi chỉ lộ ra lúc import module đã biên dịch).

    python fix_numpy_alias.py <đường dẫn ImparTail>
"""
import os, re, sys

#: np.float -> float. Không đụng np.float32/float64 nhờ ranh giới \b và (?!\d).
ALIASES = {"float": "float", "int": "int", "bool": "bool",
           "object": "object", "str": "str", "long": "int", "complex": "complex"}
PAT = re.compile(r"\bnp\.(" + "|".join(ALIASES) + r")\b(?!\d|_)")


def patch(path):
    try:
        src = open(path, encoding="utf-8").read()
    except (OSError, UnicodeDecodeError):
        return 0
    out = PAT.sub(lambda m: ALIASES[m.group(1)], src)
    if out == src:
        return 0
    open(path, "w", encoding="utf-8").write(out)
    n = len(PAT.findall(src))
    print(f"  ✓ {os.path.relpath(path):52} {n} chỗ")
    return n



_tot = 0
for _dp, _dn, _fn in os.walk(ROOT):
    if ".git" in _dp: continue
    for _f in _fn:
        if _f.endswith((".py", ".pyx")):
            _tot += patch(os.path.join(_dp, _f))
print(f"  vá {_tot} chỗ np.float/np.bool")


for sub in ("lib/draw_rectangles", "lib/fpn/box_intersections_cpu"):
    d = f"{ROOT}/{sub}"
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"],
                       cwd=d, capture_output=True, text=True)
    so = glob.glob(f"{d}/*.so")
    print(f"  {'✓' if so else '✗'} {sub}: {[os.path.basename(x) for x in so] or (r.stderr.strip().split(chr(10))[-1][:80])}")

In [ ]:
# ── 5. Vá THC rồi biên dịch CUDA extension ──────────────────────────
# Lỗi gốc lần trước:
#   ROIAlign_cuda.cu:5:10: fatal error: THC/THC.h: No such file or directory
# THC là header CUDA cũ của PyTorch, GỠ HẲN từ torch 1.11. Mã fasterRCNN viết
# thời torch 1.x nên còn include. Chỉ 5 ký hiệu THC được dùng, đều thay được:
#   THCCeilDiv -> impartail_ceil_div | THCudaCheck -> AT_CUDA_CHECK
#   THCudaMalloc/Free -> c10::cuda::CUDACachingAllocator::raw_alloc/raw_delete
import os, re, sys, subprocess

#!/usr/bin/env python3
"""Cập nhật CUDA extension của fasterRCNN cho torch hiện đại.

Vì sao cần: `THC/THC.h` là header CUDA cũ của PyTorch, **bị gỡ hẳn từ torch
1.11**. Mã fasterRCNN trong ImparTail viết thời torch 1.x nên còn include nó, và
biên dịch trên bất kỳ torch nào từ 1.11 trở lên đều dừng ở:

    ROIAlign_cuda.cu:5:10: fatal error: THC/THC.h: No such file or directory

Đây không phải lỗi khó — chỉ có năm ký hiệu THC được dùng, tất cả đều có thay
thế thẳng trong ATen:

    THC/THC.h, THCDeviceUtils.cuh  -> không cần, ATen đã có
    THC/THCAtomics.cuh            -> ATen/cuda/Atomic.cuh
    THCCeilDiv(a, b)              -> impartail_ceil_div (tự khai báo, xem HEADERS)
    THCudaCheck(x)                -> AT_CUDA_CHECK(x)
    THCudaMalloc(state, n)        -> c10::cuda::CUDACachingAllocator::raw_alloc(n)
    THCudaFree(state, p)          -> c10::cuda::CUDACachingAllocator::raw_delete(p)
    THCState *state = ...lazyInitCUDA()  -> bỏ, không còn dùng

    python fix_thc.py <đường dẫn ImparTail>
"""
import os, re, sys

#: Tự định nghĩa ceil_div thay vì dùng at::ceil_div. Lý do: ATen/ceil_div.h chỉ
#: có từ torch 2.5; trên 2.4.1 (bản duy nhất còn hỗ trợ sm_60 của Tesla P100)
#: thì at::ceil_div chưa tồn tại, và biên dịch dừng ở
#:   error: namespace "at" has no member "ceil_div"
#: Hàm chỉ có một dòng nên tự khai báo là chắc chắn nhất, không phụ thuộc bản torch.
#: Header xác minh trực tiếp trên nguồn tag v2.4.1 của pytorch, không đoán:
#:   AT_CUDA_CHECK        -> aten/src/ATen/cuda/Exceptions.h   (KHÔNG phải CUDAException.h)
#:   getCurrentCUDAStream -> c10/cuda/CUDAStream.h             (KHÔNG phải CUDAContext.h)
#:   raw_alloc/raw_delete -> c10/cuda/CUDACachingAllocator.h
#:   gpuAtomicAdd         -> aten/src/ATen/cuda/Atomic.cuh
HEADERS = """#include <ATen/cuda/CUDAContext.h>
#include <ATen/cuda/Exceptions.h>
#include <ATen/cuda/Atomic.cuh>
#include <c10/cuda/CUDAStream.h>
#include <c10/cuda/CUDACachingAllocator.h>

namespace {
template <typename T>
__host__ __device__ __forceinline__ T impartail_ceil_div(T a, T b) {
  return (a + b - 1) / b;
}
}  // namespace"""

SUBS = [
    # include cũ -> bộ header hiện đại (chỉ chèn một lần cho tệp đầu tiên khớp)
    (re.compile(r"#include <THC/THC\.h>\n(#include <THC/THCAtomics\.cuh>\n)?"
                r"(#include <THC/THCDeviceUtils\.cuh>\n)?"), HEADERS + "\n"),
    (re.compile(r"#include <THC/[A-Za-z]+\.cuh>\n"), ""),
    (re.compile(r"\bTHCCeilDiv\b"), "impartail_ceil_div"),
    (re.compile(r"\bTHCudaCheck\b"), "AT_CUDA_CHECK"),
    (re.compile(r"THCudaMalloc\(state,\s*"),
     "c10::cuda::CUDACachingAllocator::raw_alloc("),
    (re.compile(r"THCudaFree\(state,\s*"),
     "c10::cuda::CUDACachingAllocator::raw_delete("),
    # dòng khởi tạo state không còn ý nghĩa
    (re.compile(r"^\s*THCState\s*\*\s*state\s*=.*$", re.M), ""),

    # ── API Tensor cũ, bỏ trong torch 1.x → 2.x ──────────────────────
    # Kiểm trực tiếp trên nguồn tag v2.4.1: TensorBase.h không còn khai báo
    # `.type()`, và `.data<T>()` đã thay bằng `data_ptr<T>()`.
    #   x.type().is_cuda()  -> x.is_cuda()          (25 + 18 chỗ)
    #   x.data<T>()         -> x.data_ptr<T>()      (26 chỗ)
    #   a.type() == b.type()-> a.scalar_type() == b.scalar_type()
    #   AT_CHECK            -> TORCH_CHECK
    (re.compile(r"\.type\(\)\.is_cuda\(\)"), ".is_cuda()"),
    (re.compile(r"(\w+)\.type\(\)\s*==\s*(\w+)\.type\(\)"),
     r"\1.scalar_type() == \2.scalar_type()"),
    (re.compile(r"AT_DISPATCH_FLOATING_TYPES\(\s*(\w+)\.type\(\)"),
     r"AT_DISPATCH_FLOATING_TYPES(\1.scalar_type()"),
    (re.compile(r"\.data<([A-Za-z_][A-Za-z0-9_]*)>\(\)"), r".data_ptr<\1>()"),
    (re.compile(r"\bAT_CHECK\b"), "TORCH_CHECK"),
]


def patch(path):
    src = open(path, encoding="utf-8").read()
    out = src
    for pat, rep in SUBS:
        out = pat.sub(rep, out)
    if out == src:
        return 0
    open(path, "w", encoding="utf-8").write(out)
    left = len(re.findall(r"\bTHC[A-Za-z_]*\b", out))
    print(f"  ✓ {os.path.basename(path):22} còn {left} ký hiệu THC")
    return 1



ROOT = "/kaggle/working/ImparTail"
csrc = os.path.join(ROOT, "fasterRCNN/lib/model/csrc")
n = 0
for dp, _, fn in os.walk(csrc):
    for f in fn:
        if f.endswith((".cu", ".cuh", ".h", ".cpp")):
            n += patch(os.path.join(dp, f))
left = sum(len(re.findall(r"\bTHC[A-Za-z_]*\b", open(os.path.join(dp,f),encoding="utf-8").read()))
           for dp,_,fns in os.walk(csrc) for f in fns
           if f.endswith((".cu",".cuh",".h",".cpp")))
print(f"  đã vá {n} tệp | còn {left} tham chiếu THC")

env = dict(os.environ, MAX_JOBS="1", TORCH_CUDA_ARCH_LIST="6.0")
r = subprocess.run([sys.executable, "setup.py", "build", "develop"],
                   cwd=f"{ROOT}/fasterRCNN/lib", capture_output=True, text=True, env=env)
full = r.stdout + r.stderr
errs = [l for l in full.split("\n")
        if re.search(r"\berror\b|fatal error|no member|undefined|not declared|"
                     r"No such file|ninja: build stopped", l, re.I)]
print(f"\n=== biên dịch: exit {r.returncode} ===")
for l in errs[:25]: print("  " + l.strip()[:150])
if not errs: print("  không có dòng lỗi nào")

chk = subprocess.run([sys.executable, "-c",
       "import sys; sys.path.insert(0,'/kaggle/working/ImparTail');"
       "from fasterRCNN.lib.model.roi_layers import nms; print('nms OK')"],
      capture_output=True, text=True, cwd=ROOT)
print("\n✓✓ nms IMPORT ĐƯỢC — rào cản chính đã gỡ" if "nms OK" in chk.stdout
      else f"\n✗ nms hỏng: {chk.stderr.strip().split(chr(10))[-1][:140]}")

In [ ]:
# ── 6. Thư viện ────────────────────────────────────────────────────
# Danh sách này do QUÉT CÂY IMPORT ở máy mà ra (32 module cục bộ từ
# test_sgg_methods.py), không phải cài thử từng cái theo lỗi.
#
# pyrebase: chỉ để tác giả đẩy kết quả lên Firebase. Dòng gọi nó ĐÃ BỊ COMMENT
# (test_sgg_base.py:158 `# self._publish_results_to_firebase()`) nhưng import ở
# đầu tệp thì còn, nên vẫn chặn. Không cài được thì vô hiệu hoá import — nó
# không liên quan gì tới đánh giá.
import subprocess, sys, os, importlib.util as iu

PKGS = ["opencv-python-headless", "dill", "easydict", "h5py", "matplotlib",
        "scikit-image", "Wand", "wandb", "torchdiffeq", "torchsde"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS,
               capture_output=True, text=True)

# torchvision phải khớp torch 2.4.1
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchvision==0.19.1", "--index-url",
                "https://download.pytorch.org/whl/cu121"], capture_output=True, text=True)

# Firebase: GỠ VÔ ĐIỀU KIỆN, không thử cài.
# pyrebase4 cài được nhưng import của nó kéo theo pyOpenSSL đời cũ và chết ở
#   AttributeError: module 'lib' has no attribute 'GEN_EMAIL'
# Kiểm ở máy: FirebaseService() chỉ được gọi trong _publish_results_to_firebase(),
# và MỌI lời gọi hàm đó đều đã bị tác giả comment
# (test_sgg_base.py:158, test_sga_base.py:210). Nên gỡ import là an toàn tuyệt đối
# — nó chỉ để đẩy kết quả lên cloud, không dính gì tới đánh giá.
for f in ("test_sgg_base.py", "test_sga_base.py", "test_easg_base.py"):
    fp = f"/kaggle/working/ImparTail/{f}"
    if not os.path.exists(fp): continue
    s_ = open(fp, encoding="utf-8").read()
    s2 = s_.replace("from analysis.results.FirebaseService import FirebaseService",
                    "FirebaseService = None  # gỡ: chỉ dùng để đẩy cloud, đã tắt sẵn")
    if s2 != s_:
        open(fp, "w", encoding="utf-8").write(s2)
        print(f"  ✓ gỡ import Firebase khỏi {f}")

print("  === kiểm import ===")
for m in ("cv2","dill","easydict","h5py","matplotlib","skimage","wand",
          "wandb","torchvision","networkx","PIL","scipy","yaml","tqdm"):
    chk = subprocess.run([sys.executable, "-c", f"import {m}"],
                         capture_output=True, text=True)
    print(f"    {'✓' if chk.returncode==0 else '✗'} {m}"
          + ("" if chk.returncode==0 else f"  {chk.stderr.strip().split(chr(10))[-1][:70]}"))

# ── Bù module `analysis` thiếu trong kho mã ─────────────────────────
# Chạy thử ở máy phát hiện: ba tệp test đều import từ analysis.results, nhưng
# thư mục analysis/ KHÔNG CÓ trong kho mã công bố (và cũng không trong
# .gitignore). Thiếu nó thì script dừng ngay ở import.
# Ba lớp đó chỉ dùng trong _prepare_metrics_from_stats() và
# _publish_results_to_firebase() — nhánh Firebase đã bị comment sẵn — nên module
# giả không đổi con số đánh giá nào.
#!/usr/bin/env python3
"""Tạo module `analysis` còn thiếu trong kho mã ImparTail.

Ba tệp test (`test_sgg_base.py`, `test_sga_base.py`, `test_easg_base.py`) đều
import từ `analysis.results`, nhưng **thư mục `analysis/` không có trong kho mã
đã công bố** và cũng không nằm trong `.gitignore`. Thiếu nó thì mọi script đánh
giá dừng ngay ở dòng import, trước khi nạp được mô hình.

Kiểm ở máy cho thấy `Result`, `ResultDetails`, `Metrics` chỉ được dùng bên trong
`_prepare_metrics_from_stats()` và `_publish_results_to_firebase()` — cả hai đều
thuộc nhánh đẩy kết quả lên Firebase, mà mọi lời gọi nhánh đó đã bị tác giả
comment (`test_sgg_base.py:158`, `test_sga_base.py:210`).

Nên module giả chỉ cần thoả import. Nó **không** thay đổi con số nào của phép
đánh giá; nếu có mã nào thật sự dùng tới thì các lớp này vẫn ghi nhận thuộc tính
bình thường thay vì lỗi âm thầm.

    python stub_analysis.py <đường dẫn ImparTail>
"""
import os, sys

RESULT_PY = '''"""Bù cho module `analysis.results` không có trong kho mã công bố.

Chỉ đủ để thoả import. Xem impartail/patches/stub_analysis.py để biết vì sao.
"""


class _Bag:
    """Nhận mọi keyword, giữ lại, và cho gán thêm thuộc tính về sau."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def __getattr__(self, name):
        # phương thức add_* / set_* gọi được mà không nổ
        if name.startswith(("add_", "set_", "publish_")):
            return lambda *a, **k: None
        raise AttributeError(name)

    def __repr__(self):
        return f"{type(self).__name__}({self.__dict__})"


class Metrics(_Bag):
    pass


class ResultDetails(_Bag):
    pass


class Result(_Bag):
    pass
'''

FIREBASE_PY = '''"""Bù cho `analysis.results.FirebaseService`.

Dịch vụ thật đẩy kết quả lên Firebase và cần `pyrebase`, thư viện kéo theo một
bản `pyOpenSSL` cũ không chạy được với `cryptography` hiện tại. Mọi lời gọi nó
trong kho mã đã bị comment sẵn, nên bản giả này chỉ để import không lỗi.
"""


class FirebaseService:
    def __init__(self, *a, **k):
        raise RuntimeError(
            "FirebaseService bị vô hiệu hoá: chỉ dùng để đẩy kết quả lên cloud, "
            "không liên quan tới đánh giá."
        )
'''


def main(root):
    d = os.path.join(root, "analysis", "results")
    if os.path.isdir(os.path.join(root, "analysis")) and not os.environ.get("FORCE"):
        print("  analysis/ đã tồn tại — không ghi đè")
        return 0
    os.makedirs(d, exist_ok=True)
    for pkg in (os.path.join(root, "analysis"), d):
        open(os.path.join(pkg, "__init__.py"), "w").close()
    open(os.path.join(d, "Result.py"), "w", encoding="utf-8").write(RESULT_PY)
    open(os.path.join(d, "FirebaseService.py"), "w", encoding="utf-8").write(FIREBASE_PY)
    print(f"  ✓ tạo analysis/results/ với Result.py và FirebaseService.py")
    return 0



main("/kaggle/working/ImparTail")


In [ ]:
# ── 7. Checkpoint: tải thẳng từ UT Dallas Box ───────────────────────
# Box KHÔNG chặn tải tự động — endpoint box_download_shared_file dùng được,
# đã kiểm ở máy: 3 tệp, 1,1 GB, đúng định dạng torch pickle.
#   sttran_predcls / sttran_sgcls : sharedName ji6aek…, file_id f_16742…
#   faster_rcnn_ag                : sharedName 1pspm5…, file_id f_16600…
import os, subprocess, glob
CK = "/kaggle/working/ckpt"; os.makedirs(CK, exist_ok=True)
BOX = "https://utdallas.box.com/index.php?rm=box_download_shared_file"
FILES = [
  ("faster_rcnn_ag.pth",  "1pspm5x8etlczoklyw4bclotsmlejagj", "f_1660078888773"),
  ("sttran_predcls.tar",  "ji6aekd4d8iguqk59rfwcalhpu608mop", "f_1674218919601"),
  ("sttran_sgcls.tar",    "ji6aekd4d8iguqk59rfwcalhpu608mop", "f_1674209087058"),
]
for name, shared, fid in FILES:
    dst = f"{CK}/{name}"
    if os.path.exists(dst) and os.path.getsize(dst) > 1e8:
        print(f"  đã có {name} ({os.path.getsize(dst)/1e6:.0f} MB)"); continue
    subprocess.run(["curl", "-sL", "-o", dst, "--max-time", "600",
                    f"{BOX}&shared_name={shared}&file_id={fid}"], check=False)
    sz = os.path.getsize(dst) if os.path.exists(dst) else 0
    print(f"  {'✓' if sz > 1e8 else '✗'} {name}: {sz/1e6:.0f} MB")

# fasterRCNN đọc checkpoint ở đường dẫn cố định trong lib/object_detector.py
import re as _re
od = "/kaggle/working/ImparTail/lib/object_detector.py"
src = open(od, encoding="utf-8").read()
src2 = _re.sub(r'["\'][^"\']*faster_rcnn_ag\.pth["\']', f'"{CK}/faster_rcnn_ag.pth"', src)
if src2 != src:
    open(od, "w", encoding="utf-8").write(src2)
    print("  ✓ trỏ object_detector.py sang checkpoint vừa tải")
print("  ckpt:", sorted(os.listdir(CK)))

In [ ]:
# ── 8. CHẨN ĐOÁN — chạy được tới đâu ────────────────────────────────
import os, glob, importlib, traceback
rows = []
def chk(name, fn):
    try: ok, note = fn()
    except Exception as e: ok, note = False, f"{type(e).__name__}: {str(e)[:60]}"
    rows.append((name, ok, note)); return ok

chk("khung hình đã gắn",
    lambda: (len(os.listdir(f"{AG}/frames")) > 100,
             f"{len(os.listdir(f'{AG}/frames'))} video"))
_ANN_NEEDED = ["person_bbox.pkl", "object_bbox_and_relationship.pkl",
               "object_classes.txt", "relationship_classes.txt"]
chk("annotation đã gắn",
    lambda: (all(os.path.exists(f"{AG}/annotations/{f}") for f in _ANN_NEEDED),
             ", ".join(f for f in _ANN_NEEDED
                       if not os.path.exists(f"{AG}/annotations/{f}")) or "đủ 4 tệp"))
chk("Cython (2 extension)",
    lambda: (bool(glob.glob("/kaggle/working/ImparTail/lib/draw_rectangles/*.so")) and
             bool(glob.glob("/kaggle/working/ImparTail/lib/fpn/box_intersections_cpu/*.so")),
             "draw_rectangles + bbox"))
def _nms():
    # PHẢI kiểm qua subprocess. Tiến trình notebook đã nạp torch 2.10 từ lúc
    # khởi động; ô 0b cài torch 2.4.1 nhưng bản cũ vẫn nằm trong bộ nhớ. Extension
    # biên dịch cho 2.4.1 sẽ không nạp được vào tiến trình đang giữ 2.10 —
    # import ở đây cho kết quả SAI (đã bị đúng lỗi này ở v6).
    import subprocess, sys
    r = subprocess.run([sys.executable, "-c",
         "import sys; sys.path.insert(0,'/kaggle/working/ImparTail');"
         "from fasterRCNN.lib.model.roi_layers import nms; print('OK')"],
        capture_output=True, text=True, cwd="/kaggle/working/ImparTail")
    if "OK" in r.stdout:
        return True, "import OK (subprocess)"
    return False, (r.stderr.strip().split("\n")[-1][:100] if r.stderr else "lỗi không rõ")
chk("fasterRCNN CUDA ext", _nms)
chk("checkpoint tác giả",
    lambda: (bool(glob.glob("/kaggle/working/ckpt/*.tar") + glob.glob("/kaggle/working/ckpt/*.pth")), "dataset impartail-ckpt-ag"))

print(f"  {'bước':<26} {'':3} ghi chú")
for n, ok, note in rows:
    print(f"  {n:<26} {'✓' if ok else '✗':3} {note}")

blockers = [n for n, ok, _ in rows if not ok]
print()
if blockers:
    print("CHẶN Ở:", ", ".join(blockers))
    print("\nKhông chạy được đánh giá. Đây là kết quả có giá trị — nó cho biết")
    print("chính xác rào cản nằm ở đâu, thay vì phỏng đoán.")
else:
    print("Mọi thứ sẵn sàng — chạy ô tiếp theo.")

In [ ]:
# ── 8b. Đối soát khung hình test với ảnh thực có ───────────────────
# dataloader/standard/action_genome/ag_dataset.py:22 dùng cv2.imread, hàm này
# trả None khi thiếu tệp thay vì báo lỗi; dòng kế prep_im_for_blob(im, ...) sẽ
# nổ AttributeError 'NoneType' has no attribute 'shape' — có thể sau khi đã
# chạy hàng nghìn video. Nên đối soát TRƯỚC, và báo số liệu thay vì bỏ qua
# âm thầm: thiếu khung hình thì tập đánh giá không còn so sánh được với bài báo.
import pickle, os, collections

with open(f"{AG}/annotations/object_bbox_and_relationship.pkl", "rb") as f:
    obj = pickle.load(f)

test_frames = [k for k, v in obj.items() if v[0]["metadata"]["set"] == "test"]
vids = collections.Counter(k.split("/")[0] for k in test_frames)
print(f"khung hình test theo annotation : {len(test_frames):,}")
print(f"video test theo annotation      : {len(vids):,}")

have = set(os.listdir(f"{AG}/frames"))
missing_vid = [v for v in vids if v not in have]
print(f"video test THIẾU thư mục        : {len(missing_vid):,}")

# lấy mẫu 400 khung hình để ước lượng tỉ lệ thiếu ảnh mà không phải stat 130k tệp
import random
random.seed(0)
sample = random.sample(test_frames, min(400, len(test_frames)))
miss = [n for n in sample if not os.path.exists(f"{AG}/frames/{n}")]
print(f"mẫu 400 khung hình, thiếu ảnh   : {len(miss)}  ({100*len(miss)/len(sample):.1f}%)")
if miss:
    print("  ví dụ:", miss[:3])

DATA_OK = not (missing_vid or miss)
if not DATA_OK:
    print("\n⚠ Dữ liệu KHÔNG đủ cho tập test đầy đủ.")
    print("  v7 đã chứng minh: chạy tiếp thì crash ở cv2.imread sau 553/1750 video")
    print("  (AttributeError: 'NoneType' has no attribute 'astype',")
    print("   ag_dataset.py:25 -> blob.py:39). Ô 9 sẽ KHÔNG chạy.")
else:
    print("\n✓ Đủ toàn bộ khung hình test — kết quả sẽ so sánh được với bài báo.")


In [ ]:
# ── 9. Đánh giá (chỉ chạy khi ô 8 báo sẵn sàng) ────────────────────
# Tên tham số lấy nguyên từ lib/supervised/config.py + run_sgg_predcls_image.sh
# của tác giả, KHÔNG tự đặt:
#   --ckpt (không phải --model_path)  · config.py:80
#   --task_name sgg   (mặc định là "sga" -> phải ghi rõ)  · config.py:88
#   --method_name sttran                                   · config.py:73
#   --save_path  : mặc định /data/rohith/ag/checkpoints, stsg_base.py:41 gọi
#                  os.makedirs lên đó -> PermissionError trên Kaggle. Bắt buộc đổi.
#   bỏ --use_wandb (store_true, không cần), bỏ --use_input_corruptions
#   (đánh giá VidSGG sạch trước, corruption để sau).
#
# stsg_base.py:46-58 tách tên checkpoint theo quy ước sttran_<mode>_epoch_<n>.tar:
#   name = basename.split('.')[0]; checkpoint_name = "_".join(name.split('_')[:-2])
# Với "sttran_predcls.tar" -> [:-2] cho danh sách RỖNG -> tên rỗng -> tệp kết quả
# thành "with_constraint_.csv" và nhánh mode không nhận ra được. Vì vậy đổi tên
# bản sao theo đúng quy ước trước khi truyền vào.
import glob, os, shutil, subprocess, sys

SRC = "/kaggle/working/ckpt/sttran_predcls.tar"
CKPT = "/kaggle/working/ckpt/sttran_predcls_epoch_3.tar"   # đúng quy ước tác giả
if os.path.exists(SRC) and not os.path.exists(CKPT):
    shutil.copy(SRC, CKPT)

# DATA_OK do ô 8b đặt. Thiếu cờ này nghĩa là chưa đối soát -> không chạy:
# v7 chạy 20 phút rồi crash vì bỏ qua đúng cảnh báo đó.
ready = (bool(glob.glob("/kaggle/working/ImparTail/lib/draw_rectangles/*.so"))
         and os.path.exists(CKPT)
         and globals().get("DATA_OK", False))
if not ready:
    print("bỏ qua — ô 8/8b còn rào cản chưa gỡ (xem DATA_OK ở trên)")
else:
    os.makedirs("/kaggle/working/save", exist_ok=True)
    # PREDCLS rẻ nhất: cho sẵn hộp bao và nhãn vật thể, chỉ phân loại vị ngữ.
    # predcls cũng là mode duy nhất né được dataloader/..._filtersmall.pkl
    # (base_ag_dataset.py:94 mở tệp đó bằng đường dẫn tương đối; test_sgg_base.py:305
    #  chỉ đặt filter_small_box=True khi mode != predcls).
    cmd = [sys.executable, "test_sgg_methods.py",
           "--task_name", "sgg",
           "--method_name", "sttran",
           "--mode", "predcls",
           "--datasize", "large",
           "--data_path", "/kaggle/working/data/ag",
           "--save_path", "/kaggle/working/save",
           "--ckpt", CKPT]
    print(" ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd="/kaggle/working/ImparTail",
                       capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(out[-40:]))
    print(f"\nexit code: {r.returncode}")

    for f in glob.glob("/kaggle/working/ImparTail/results/**/*.csv", recursive=True):
        print("kết quả:", f)
        print(open(f).read()[:1500])
